In [41]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model
import warnings
warnings.filterwarnings('ignore')
df_test = pd.read_csv("final_.csv")
df_test["Timestamp"] = pd.to_datetime(df_test["Timestamp"])
# Handle missing values
df_test.fillna(method="ffill", inplace=True)
df_test.fillna(method="bfill", inplace=True)

# Select target and features
target = "AQI"
features = [
    "PM2.5 (µg/m³)",
    "PM10 (µg/m³)",
    "Ozone (µg/m³)",
    "NO2 (µg/m³)",
    "NO (µg/m³)",
    "SO2 (µg/m³)",
    "CO (mg/m³)",
    "NH3 (µg/m³)",
]
df_test = df_test[["Timestamp"] + features + [target]]
# rows = []

# for i in range(11):
#     row = {
#         "PM2.5 (µg/m³)":np.random.randint(300,900),
#         "PM10 (µg/m³)": np.random.randint(300,900),
#         "Ozone (µg/m³)": np.random.randint(300,900),
#         "NO2 (µg/m³)": np.random.randint(300,900),
#         "NO (µg/m³)": np.random.randint(300,900),
#         "SO2 (µg/m³)":np.random.randint(300,900),
#         "CO (mg/m³)": np.random.randint(300,900),
#         "NH3 (µg/m³)": np.random.randint(300,900)
#     }
#     rows.append(row)
# df_forecast = pd.DataFrame(rows)
# df_forecast['Timestamp']=df_test["Timestamp"]
# df_test=df_forecast
df_test=df_test[4:14]
df_test

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),Ozone (µg/m³),NO2 (µg/m³),NO (µg/m³),SO2 (µg/m³),CO (mg/m³),NH3 (µg/m³),AQI
4,2019-01-05,76.57,202.300,63.44,64.11,61.42,30.220,3.230,5.543333,178.0
5,2019-01-06,80.92,201.085,61.47,60.19,49.21,28.205,2.330,5.543333,192.0
6,2019-01-07,53.07,152.360,48.66,69.04,43.16,29.815,2.100,5.543333,146.0
7,2019-01-08,56.07,140.085,67.36,56.31,52.55,34.095,2.325,5.543333,161.0
8,2019-01-09,74.09,216.010,85.43,48.68,57.36,28.300,2.210,5.543333,154.0
9,2019-01-10,84.82,233.730,69.76,51.30,72.74,27.880,2.575,5.543333,219.0
10,2019-01-11,83.62,243.860,71.35,46.42,75.99,34.040,2.615,5.543333,212.0
11,2019-01-12,78.65,238.240,79.40,49.48,57.38,34.840,2.490,5.543333,200.0
12,2019-01-13,74.26,206.340,75.82,34.15,62.04,34.495,2.615,5.543333,206.0
13,2019-01-14,66.22,177.290,72.51,44.56,36.98,32.490,2.320,5.543333,163.0


In [42]:
class TLSTMCell(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super(TLSTMCell, self).__init__(**kwargs)
        self.units = units
        self.lstm_cell = tf.keras.layers.LSTMCell(units)

    @property
    def state_size(self):
        return self.lstm_cell.state_size

    @property
    def output_size(self):
        return self.lstm_cell.output_size

    def call(self, inputs, states, training=None):
        # Assume inputs shape: (batch, features + 1) where the last channel is delta_t.
        feature_dim = tf.shape(inputs)[-1] - 1
        features = inputs[:, :feature_dim]
        delta_t = inputs[:, feature_dim:]
        # Apply time decay to features
        adjusted_features = features * tf.exp(-delta_t)
        return self.lstm_cell(adjusted_features, states, training=training)

    def get_config(self):
        config = super(TLSTMCell, self).get_config()
        config.update(
            {
                "units": self.units,
            }
        )
        return config

In [43]:
with open("./training_data/scaler_features.pkl", "rb") as f:
    loaded_scaler = pickle.load(f)
    df_test[features] = loaded_scaler.fit_transform(df_test[features])
    
with open("./training_data/scaler_target.pkl", "rb") as f:
    scaler_target = pickle.load(f)
# Prepare the last 10-day input window
X_features = np.array(df_test[features].iloc[-10:].values).reshape(1, 10, len(features))

# Compute time intervals (Δt in days)
timestamps = df_test["Timestamp"].reset_index(drop=True)
ts_window = pd.to_datetime(timestamps.iloc[-10:])
dt = [0]  # First time difference is always 0
dt += list(ts_window.diff().fillna(pd.Timedelta(days=0)).dt.days.values[1:])
X_time = np.array(dt).reshape(1, 10, 1).astype("float32")


model = load_model("training_data/model", custom_objects={"TLSTMCell": TLSTMCell})

# Predict AQI for the next day
predicted_scaled_aqi = model.predict([X_features, X_time])

# Convert back to original AQI scale
# predicted_aqi = scaler1.inverse_transform([[0] * len(features) + [predicted_scaled_aqi[0][0]]])[0][-1]
print(f"Predicted AQI for Next Day: {predicted_scaled_aqi}")


1/1 [==============================] - 0s 335ms/step
Predicted AQI for Next Day: [[0.33392602]]


In [44]:
scaler_target.inverse_transform(predicted_scaled_aqi)[0][0]

170.3766